# Kernel-aware, sensitivity-ranked forward selection

This example assigns each perturbation set independently to the linear and RBF leaves of a mixed kernel. Forward selection uses frozen-model decision sensitivity through `feature_ranker`, rather than choosing every later addition by its cross-validation score.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mistic import MixedKernel, cvSet, kernelWrapper, paramSet, score_svc, svmSet

In [2]:
data = load_breast_cancer()
X_dev, X_blind, y_dev, y_blind = train_test_split(
    data.data[:, :6], data.target, test_size=0.25, stratify=data.target, random_state=7
)
scaler = StandardScaler().fit(X_dev)
X_dev = scaler.transform(X_dev)
X_blind = scaler.transform(X_blind)

splits = cvSet(X_dev, y_dev)
splits.classification(num_sets=3, random_seed=7)

Each entry in `perturbation_sets` is indivisible. MISTIC creates one selection unit per `(kernel_id, perturbation_set)` assignment, so removing a group from `radial` does not remove it from `linear`.

In [3]:
kernel = MixedKernel.weighted_sum([
    kernelWrapper("linear", name="linear"),
    kernelWrapper("rbf", name="radial"),
], weights=[0.5, 0.5])

ensemble = svmSet(
    SVC(kernel="precomputed", class_weight="balanced"),
    splits, score_svc().score, kernel=kernel,
    kernel_feature_selection="independent",
    perturbation_sets=[[0, 2], [1], [3, 5], [4]],
)
pd.DataFrame([
    {"unit": i, "kernel_id": unit.kernel_id, "features": unit.features, "group": unit.name}
    for i, unit in enumerate(ensemble.selection_space_)
])

,unit,kernel_id,features,group
0,0,linear,"(0, 2)",group_0
1,1,linear,"(1,)",group_1
2,2,linear,"(3, 5)",group_2
3,3,linear,"(4,)",group_3
4,4,radial,"(0, 2)",group_0
5,5,radial,"(1,)",group_1
6,6,radial,"(3, 5)",group_2
7,7,radial,"(4,)",group_3


In [4]:
grid = [
    paramSet({"C": C}, {"radial__gamma": gamma})
    for C in (0.5, 2.0) for gamma in (0.01, 0.1)
]
ensemble.greedy_forward_kernel_selection(
    grid,
    max_kernel_features=8,
    num_initial_kernel_features=1,
    addition_factor=0.35,
    feature_ranker=ensemble.kernel_feature_sensitivity_rank,
    set_for_rank="train",
)

Kernel features: 2, unique features: 2, Score: 0.917
Kernel features: 5, unique features: 5, Score: 0.950
Kernel features: 6, unique features: 6, Score: 0.964
Kernel features: 7, unique features: 6, Score: 0.968


Kernel features: 8, unique features: 6, Score: 0.966


In [5]:
selected = pd.DataFrame([
    {"unit": i, "kernel_id": ensemble.selection_space_[i].kernel_id,
     "features": ensemble.selection_space_[i].features}
    for i in ensemble.selected_selection_units_
])
display(selected)
print("Kernel-specific features:", {k: v.tolist() for k, v in ensemble.kernel_features.items()})
print("Unique features:", ensemble.features.tolist())
pd.DataFrame(ensemble.kernel_feature_performance_).T

,unit,kernel_id,features
0,0,linear,"(0, 2)"
1,3,linear,"(4,)"
2,2,linear,"(3, 5)"
3,1,linear,"(1,)"
4,5,radial,"(1,)"
5,7,radial,"(4,)"


Kernel-specific features: {'linear': [0, 1, 2, 3, 4, 5], 'radial': [1, 4]}
Unique features: [0, 1, 2, 3, 4, 5]


,f1,auc,brier,calibration,score,C,radial__gamma,num_kernel_features,num_features,added_kernel,added_feature
0,0.889023,0.945425,NaN,NaN,0.917224,2.0,0.01,2,2,"(linear,)","((0, 2),)"
1,0.924552,0.975249,NaN,NaN,0.9499,0.5,0.01,5,5,"(linear, linear)","((4,), (3, 5))"
2,0.945637,0.981741,NaN,NaN,0.963689,2.0,0.01,6,6,"(linear,)","((1,),)"
3,0.952496,0.983161,NaN,NaN,0.967828,2.0,0.1,7,6,"(radial,)","((1,),)"
4,0.94939,0.983364,NaN,NaN,0.966377,2.0,0.1,8,6,"(radial,)","((4,),)"


In [6]:
blind_decision = ensemble.decision_function(X_blind)
print(f"Blind ROC AUC: {roc_auc_score(y_blind, blind_decision):.3f}")
print("Selected original columns:", [data.feature_names[i] for i in ensemble.features])

Blind ROC AUC: 0.988
Selected original columns: ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness']
